Summary table for business question 4.2.

In [0]:
from pyspark.sql import functions as F

In [0]:
SOURCE_CATALOG_NAME = 'beverage_sales'
SOURCE_SCHEMA_NAME = 'gold'

TARGET_CATALOG_NAME = 'beverage_sales'
TARGET_SCHEMA_NAME = 'gold'
TARGET_TABLE_NAME = 'agg_sales_brand_month'

In [0]:
df_fact_sales = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.fact_sales').alias('fs')
df_dim_brand = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_brand').alias('db')
df_dim_date = spark.table(f'{SOURCE_CATALOG_NAME}.{SOURCE_SCHEMA_NAME}.dim_date').alias('dd')

In [0]:
df_agg = (
    df_fact_sales
    .join(
        F.broadcast(df_dim_brand),
        on='brand_key',
        how='inner'
    )
    .join(
        F.broadcast(df_dim_date),
        on='date_key',
        how='inner'
    )
    .groupBy(
        'dd.year',
        'dd.month',
        'dd.month_name',
        'dd.year_month',
        'db.brand_name'
    )
    .agg(
        F.sum('fs.dollar_volume').alias('dollar_volume'),
        F.count('*').alias('record_count')
    )
)

In [0]:
df_agg\
    .write\
    .mode('overwrite')\
    .saveAsTable(f'{TARGET_CATALOG_NAME}.{TARGET_SCHEMA_NAME}.{TARGET_TABLE_NAME}')